In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import numpy as np

# 1. MultiHeadAttention Sanity Check 

In [2]:
ex = torch.randn(3, 197, 768)

In [3]:
class MultiHeadConvNNAttention(nn.Module):
    def __init__(self, 
                 d_hidden, 
                 num_heads, 
                 attention_dropout,
                 K, 
                 seq_length=197, 
                 ):
        
        super(MultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        # Core Parameters
        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.attention_dropout = attention_dropout
        self.d_k = d_hidden // num_heads

        # ConvNN Parameters
        self.K = K
        self.seq_length = seq_length
        # Linear projections for query, key, value
        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)   

        self.W_q.weight.data.fill_(2.0)
        self.W_k.weight.data.fill_(3.0)
        self.W_v.weight.data.fill_(4.0)
        self.W_o.weight.data.fill_(5.0)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads
        self.conv = nn.Conv1d(
            in_channels=self.in_channels, 
            out_channels=self.out_channels,
            kernel_size=self.K,
            stride=self.K,
            padding=0,
            groups=self.in_channels, 
            bias=False
        )
        self.conv.weight.data.fill_(1.0)
        
    def split_head(self, x): 
        batch_size, seq_length, d_hidden = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)
        
    def combine_heads(self, x): 
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden) 

    def process_heads(self, v, attn_matrix):
        B, NH, SL, DK = v.shape

        outs = []

        for i in range(NH):
            v_i = v[:, i, :, :]
            am_i = attn_matrix[:, i, :, :]
            # print("v_i shape: ", v_i.shape)
            # print("am_i shape: ", am_i.shape)

            prime_i = self._prime(v_i.transpose(1, 2), am_i, self.K)
            # print("prime_i shape: ", prime_i.shape)

            out_i = self.conv(prime_i).transpose(1, 2).unsqueeze(1)
            # print("out_i shape: ", out_i.shape)
            
            outs.append(out_i)


        out = torch.concat(outs, dim=1)
        out = self.dropout(out)
        return out 

    def _prime(self, v, qk, K):
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)
        
        # Apply softmax
        topk_values = torch.softmax(topk_values, dim=-1)

        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime
        
    
    def forward(self, x): 
        q = self.split_head(self.W_q(x)) # (B, num_heads, seq_length, d_k)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # print("q|k|v shape: ", q.shape) 

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)
        # print("attn scores shape: ", attn_matrix.shape) 
        # print() 

        attn_output = self.process_heads(v, attn_matrix)
        # print()
        # print("attn output shape: ", attn_output.shape)
        
        output = self.W_o(self.combine_heads(attn_output)) # (B, seq_length, d_hidden)
        return output

        
# with torch.no_grad():
#     convatten = MultiHeadConvNNAttention(768, 3, 0.0, 197, 197)
#     out1 = convatten(ex)
#     print()
#     print("out shape: ", out1.shape)

In [4]:

class MultiHeadConvNNAttention_Old(nn.Module):
    def __init__(self, 
                 d_hidden, 
                 num_heads, 
                 attention_dropout,
                 K, 
                 sampling_type, 
                 num_samples, 
                 sample_padding, 
                 magnitude_type, 
                 seq_length=197, 
                 coordinate_encoding=False, 
                 convolution_type='depthwise', 
                 softmax_topk_val=True
                 ):
        
        super(MultiHeadConvNNAttention_Old, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        # Core Parameters
        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.attention_dropout = attention_dropout
        self.d_k = d_hidden // num_heads

        # ConvNN Parameters
        self.K = K
        self.seq_length = seq_length

        # 3 types of sampling: all, random, spatial
        self.sampling_type = sampling_type
        self.num_samples = int(num_samples) 
        self.sample_padding = int(sample_padding) if sampling_type == 'spatial' else 0

        # Similarity Metric 
        self.magnitude_type = magnitude_type
        self.maximum = True if self.magnitude_type in ('cosine', 'matmul') else False

        # Coordinate Encoding (optional) 
        self.coordinate_encoding = coordinate_encoding
        self.coordinate_cache = {}
        
        # Linear projections for query, key, value
        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)   

        self.W_q.weight.data.fill_(2.0)
        self.W_k.weight.data.fill_(3.0)
        self.W_v.weight.data.fill_(4.0)
        self.W_o.weight.data.fill_(5.0)
        
        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = (d_hidden // num_heads) + 1 if coordinate_encoding else (d_hidden // num_heads)
        self.out_channels = (d_hidden // num_heads) 

        # Convolution Layer 
        self.convolution_type = convolution_type

        if convolution_type == 'standard':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels,
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                bias=False
            )
        elif convolution_type == 'depthwise':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels, 
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                groups=self.in_channels, 
                bias=False
            )
            self.conv.weight.data.fill_(1.0)
        elif convolution_type == 'depthwise-separable':
            self.conv = nn.Sequential(
                # Depthwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.in_channels,
                    kernel_size=self.K,
                    stride=self.K,
                    padding=0,
                    groups=self.in_channels,
                    bias=False
                ), 
                # Pointwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0, 
                    bias=False
                )
            )

        # Softmax 
        self.softmax_topk_val = softmax_topk_val
        
        # Utility Variables 
        self.INF = 1.1
        self.NEG_INF = -0.1 
        
    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size()
        self.batch_size = batch_size
        return x.contiguous().view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
    
    def batch_combine(self, x):
        batch_size, _, seq_length, d_k = x.size()
        x = x.permute(0, 1, 3, 2).contiguous() 
        return x.view(-1, self.d_k, seq_length)

    def batch_split(self, x):
        if self.num_heads == 1:
            return x.unsqueeze(1)
        else:
            x = x.reshape(self.batch_size, -1, self.d_k, self.seq_length)
            return x.permute(0, 1, 3, 2).contiguous()

    def combine_heads(self, x):
        if self.num_heads == 1:
            return x.squeeze(1) 
        else:
            batch_size, _, seq_length, d_k = x.size()
            return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)
        
    def forward(self, x):
        # Note: x shape: (B, seq_length, d_hidden)
        # 1. Splithead & Batch Combine
        k = self.batch_combine(self.split_head(self.W_k(x)))
        v = self.batch_combine(self.split_head(self.W_v(x)))
        

        # 2. Add Coordinate Encoding 
        k = self._add_coordinate_encoding(k) if self.coordinate_encoding else k
        v = self._add_coordinate_encoding(v) if self.coordinate_encoding else v

        # 3. Sampling & Similarity Calculation
        if self.sampling_type == 'all': # All Samples
            q = self.batch_combine(self.split_head(self.W_q(x)))
            
            q = self._add_coordinate_encoding(q) if self.coordinate_encoding else q

            similarity_matrix = self._calculate_matmul_matrix(k, q) if self.magnitude_type == 'matmul' else self._calculate_cosine_matrix(k, q) if self.magnitude_type == 'cosine' else self._calculate_euclidean_matrix(k, q, sqrt=True)

            
            prime = self._prime(v, similarity_matrix, self.K, self.maximum) if not self.softmax_topk_val else self._prime_softmax(v, similarity_matrix, self.K, self.maximum)

        elif self.sampling_type == 'random': # Random Samples
            rand_idx = torch.randperm(x.shape[1], device=x.device)[:self.num_samples]
            x_sample = x[:, rand_idx, :]            
            q = self.batch_combine(self.split_head(self.W_q(x_sample)))
            q = self._add_coordinate_encoding(q) if self.coordinate_encoding else q

            similarity_matrix = self._calculate_matmul_matrix(k, q) if self.magnitude_type == 'matmul' else self._calculate_cosine_matrix(k, q) if self.magnitude_type == 'cosine' else self._calculate_euclidean_matrix(k, q, sqrt=True)

            range_idx = torch.arange(len(rand_idx), device=q.device)
            similarity_matrix[:, rand_idx, range_idx] = self.INF if self.magnitude_type == 'euclidean' else self.NEG_INF

            prime = self._prime_N(v, similarity_matrix, self.K, rand_idx, self.maximum) if not self.softmax_topk_val else self._prime_softmax_N(v, similarity_matrix, self.K, rand_idx, self.maximum)

        elif self.sampling_type == 'spatial': # Spatial Samples
            spat_idx = torch.linspace(0 + self.sample_padding, x.shape[1] - self.sample_padding - 1, self.num_samples, device=x.device).long()
            x_sample = x[:, spat_idx, :]
            q = self.batch_combine(self.split_head(self.W_q(x_sample)))
            q = self._add_coordinate_encoding(q) if self.coordinate_encoding else q

            similarity_matrix = self._calculate_matmul_matrix(k, q) if self.magnitude_type == 'matmul' else self._calculate_cosine_matrix(k, q) if self.magnitude_type == 'cosine' else self._calculate_euclidean_matrix(k, q, sqrt=True)
            
            range_idx = torch.arange(len(spat_idx), device=q.device)
            similarity_matrix[:, spat_idx, range_idx] = self.INF if self.magnitude_type == 'euclidean' else self.NEG_INF

            prime = self._prime_N(v, similarity_matrix, self.K, spat_idx, self.maximum) if not self.softmax_topk_val else self._prime_softmax_N(v, similarity_matrix, self.K, spat_idx, self.maximum)
            
        else: 
            raise ValueError("Invalid sampling_type. Must be one of ['all', 'random', 'spatial']")

        # 4. Conv1d Layer
        x = self.conv(prime)  

        # 5. Dropout + Reshape (B, seq_length, d_hidden)
        x = self.dropout(x)
        x = x.permute(0, 2, 1) 

        # 6. Final Linear Projection
        x = self.W_o(self.combine_heads(self.batch_split(x)))
        return x       

    def _calculate_matmul_matrix(self, K, Q):
        attn_matrix = torch.matmul(K.transpose(1, 2), Q) / self.d_k ** 0.5
        return attn_matrix
        
    def _calculate_euclidean_matrix(self, K, Q, sqrt=False):
        k_norm_squared = torch.sum(K**2, dim=1, keepdim=True)
        q_norm_squared = torch.sum(Q**2, dim=1, keepdim=True)
        dot_product = torch.bmm(K.transpose(1, 2), Q)

        dist_matrix = k_norm_squared.transpose(1, 2) + q_norm_squared - 2 * dot_product
        dist_matrix = torch.sqrt(dist_matrix) if sqrt else dist_matrix
        return dist_matrix 

    def _calculate_cosine_matrix(self, K, Q):
        k_norm = F.normalize(K, p=2, dim=1)
        q_norm = F.normalize(Q, p=2, dim=1)
        similarity_matrix = torch.matmul(k_norm.transpose(1, 2), q_norm)
        return similarity_matrix

    def _prime(self, v, qk, K, maximum):
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=maximum)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)

        # Normalize by K for distance metrics 
        if not maximum: 
            prime = prime / (topk_values_exp + 1e-8)
        else:
            prime = topk_values_exp * prime 

        prime = prime.view(b, c, -1)

        return prime

    def _prime_N(self, v, qk, K, rand_idx, maximum):
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K-1, dim=2, largest=maximum)
        tk = topk_indices.shape[-1]
        assert K == tk + 1, "Error: K must be same as tk + 1. K == tk + 1."

        # Map sample indicies back to original matrix positions 
        mapped_tensor = rand_idx[topk_indices]
        token_indices = torch.arange(t, device=v.device).view(1, t, 1).expand(b, t, 1)
        final_indices = torch.cat([token_indices, mapped_tensor], dim=-1)
        topk_indices_exp = final_indices.unsqueeze(1).expand(b, c, t, K)

        # Expand topk values to match the shape of indices
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K-1)
        ones = torch.ones((b, c, t, 1), device=v.device)
        topk_values_exp = torch.cat((ones, topk_values_exp), dim=-1)

        # Gather matrix values and apply similarity weighting 
        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()    
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        
        if not maximum:  # euclidean distance
            prime = prime / (topk_values_exp + 1e-8)
        else:
            prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def _prime_softmax(self, v, qk, K, maximum):
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=maximum)
        
        # Apply softmax
        if maximum:  # cosine/matmul (maximize)
            topk_values = torch.softmax(topk_values, dim=-1)
        else:  # euclidean (minimize) - negate before softmax
            topk_values = torch.softmax(-topk_values, dim=-1)

        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def _prime_softmax_N(self, v, qk, K, rand_idx, maximum):
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K-1, dim=2, largest=maximum)
        tk = topk_indices.shape[-1]
        assert K == tk + 1, "Error: K must be same as tk + 1. K == tk + 1."

        # Map sample indices back to original matrix positions 
        mapped_tensor = rand_idx[topk_indices]
        token_indices = torch.arange(t, device=v.device).view(1, t, 1).expand(b, t, 1)
        final_indices = torch.cat([token_indices, mapped_tensor], dim=-1)
        topk_indices_exp = final_indices.unsqueeze(1).expand(b, c, t, K)

        # Expand topk values to match the shape of indices
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K-1)
        ones = torch.ones((b, c, t, 1), device=v.device)
        topk_values_exp = torch.cat((ones, topk_values_exp), dim=-1)
        
        # Apply softmax
        if maximum:
            topk_values_exp = torch.softmax(topk_values_exp, dim=-1)
        else:
            topk_values_exp = torch.softmax(-topk_values_exp, dim=-1)
                
        # Gather matrix values and apply similarity weighting 
        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()    
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime

        prime = prime.view(b, c, -1)
        return prime

    def _add_coordinate_encoding(self, x):
        b, c, t = x.shape 
        cache_key = f"{b}_{t}_{x.device}"
        if cache_key in self.coordinate_cache: 
            expanded_coords = self.coordinate_cache[cache_key]
        else: 
            coords_vec = torch.linspace(start=-1, end=1, steps=t, device=x.device).unsqueeze(0).expand(b, -1) 
            expanded_coords = coords_vec.unsqueeze(1).expand(b, -1, -1) 
            self.coordinate_cache[cache_key] = expanded_coords

        x_with_coords = torch.cat([x, expanded_coords], dim=1) 
        return x_with_coords 

In [5]:

# with torch.no_grad():
#     convatten_old = MultiHeadConvNNAttention_Old(
#         768,
#         3, 
#         0.0, 
#         197, 
#         "all", 
#         -1, 
#         0, 
#         "matmul", 
#         seq_length=197)
#     out2 = convatten_old(ex)
#     print()
#     print("out shape: ", out2.shape)

# print(convatten_old.maximum)

In [6]:

"""Multi-Head Layers for Transformer Encoder"""
class MultiHeadAttention(nn.Module): 
    def __init__(self, d_hidden, num_heads, attention_dropout):
        super(MultiHeadAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"
        
        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.d_k = d_hidden // num_heads # dimension of each head
        self.dropout = nn.Dropout(attention_dropout)
        
        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)       
        self.W_q.weight.data.fill_(2.0)
        self.W_k.weight.data.fill_(3.0)
        self.W_v.weight.data.fill_(4.0)
        self.W_o.weight.data.fill_(5.0)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        attn_probs = self.dropout(torch.softmax(attn_scores, dim=-1))
        output = torch.matmul(attn_probs, V)
        return output, attn_probs
    
    def split_head(self, x): 
        batch_size, seq_length, d_hidden = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)
        
    def combine_heads(self, x): 
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden) 
    
    def forward(self, x, mask=None):
        q = self.split_head(self.W_q(x)) # (B, num_heads, seq_length, d_k)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))
        
        attn_output, _ = self.scaled_dot_product_attention(q, k, v, mask) # (B, num_heads, seq_length, d_k)
        output = self.W_o(self.combine_heads(attn_output)) # (B, seq_length, d_hidden)
        return output


with torch.no_grad():
    mha = MultiHeadAttention(768, 3, 0.0)
    out3 = mha(ex)


## Claude implementation

In [7]:


class MultiHeadConvNNAttention_Claude(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K, 
                 seq_length=197):

        super(MultiHeadConvNNAttention_Claude, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads

        self.conv = nn.Conv1d(
            in_channels=self.in_channels, 
            out_channels=self.out_channels,
            kernel_size=self.K,
            stride=self.K,
            padding=0,
            groups=self.in_channels, 
            bias=False
        )
        self.conv.weight.data.fill_(1.0)

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size() 
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def _prime(self, v, qk, K):
        # v: (B*num_heads, d_k, seq_length), qk: (B*num_heads, seq_length, seq_length)
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)

        topk_values = torch.softmax(topk_values, dim=-1)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Attention Matrix: (B, NH, SL, SL) - Q @ K^T
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Merge B and num_heads into dim for prime & conv 
        ## (B, NH, SL, DK) → (B*NH, DK, SL) for v and (B, NH, SL, SL) → (B*NH, SL, SL)
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        # Prime and Convolution 
        prime = self._prime(v_merged, am_merged, self.K)
        out = self.conv(prime) # (B*num_heads, d_k, seq_length) 

        # Reshape back: (B*NH, DK, SL) → (B, NH, SL, DK)
        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out) 

        # Combine Heads and Final Linear Projection
        output = self.W_o(self.combine_heads(out)) # (B, SL, d_hidden)
        return output

    

        

## When num_heads = 1

In [8]:
with torch.no_grad():
    convatten_new = MultiHeadConvNNAttention(768, 1, 0.0, 197, 197)
    convatten_claude = MultiHeadConvNNAttention_Claude(768, 1, 0.0, 197, 197)
    convatten_old = MultiHeadConvNNAttention_Old(768, 1, 0.0, 197, "all", -1, 0, "matmul", 197)
    mha = MultiHeadAttention(768, 1, 0.0)
    
    # Copy weights from convatten_new to all others
    for model in [convatten_claude, convatten_old, mha]:
        model.W_q.weight.data = convatten_new.W_q.weight.data.clone()
        model.W_k.weight.data = convatten_new.W_k.weight.data.clone()
        model.W_v.weight.data = convatten_new.W_v.weight.data.clone()
        model.W_o.weight.data = convatten_new.W_o.weight.data.clone()
    
    # Copy conv weights to conv-based models
    convatten_claude.conv.weight.data = convatten_new.conv.weight.data.clone()
    convatten_old.conv.weight.data = convatten_new.conv.weight.data.clone()

    out1 = convatten_new(ex)
    out2 = convatten_old(ex)
    out3 = convatten_claude(ex)
    out4 = mha(ex)

    print("new vs old:              ", torch.allclose(out1, out2))
    print("new vs claude:           ", torch.allclose(out1, out3))
    print("old vs claude:           ", torch.allclose(out2, out3))
    print()
    print("new vs mha:              ", torch.allclose(out1, out4))
    print("old vs mha:              ", torch.allclose(out2, out4))
    print("claude vs mha:           ", torch.allclose(out3, out4))
    print()
    print("Max diff (new vs old):   ", torch.max(torch.abs(out1 - out2)).item())
    print("Max diff (new vs claude):", torch.max(torch.abs(out1 - out3)).item())
    print("Max diff (old vs claude):", torch.max(torch.abs(out2 - out3)).item())

new vs old:               True
new vs claude:            True
old vs claude:            True

new vs mha:               True
old vs mha:               True
claude vs mha:            True

Max diff (new vs old):    0.0
Max diff (new vs claude): 0.0
Max diff (old vs claude): 0.0


### When num_heads = 12, d_hidden = 768

In [9]:
with torch.no_grad():
    convatten_new = MultiHeadConvNNAttention(768, 12, 0.0, 197, 197)
    convatten_claude = MultiHeadConvNNAttention_Claude(768, 12, 0.0, 197, 197)
    convatten_old = MultiHeadConvNNAttention_Old(768, 12, 0.0, 197, "all", -1, 0, "matmul", 197)
    mha = MultiHeadAttention(768, 12, 0.0)
    
    # Copy weights from convatten_new to all others
    for model in [convatten_claude, convatten_old, mha]:
        model.W_q.weight.data = convatten_new.W_q.weight.data.clone()
        model.W_k.weight.data = convatten_new.W_k.weight.data.clone()
        model.W_v.weight.data = convatten_new.W_v.weight.data.clone()
        model.W_o.weight.data = convatten_new.W_o.weight.data.clone()
    
    # Copy conv weights to conv-based models
    convatten_claude.conv.weight.data = convatten_new.conv.weight.data.clone()
    convatten_old.conv.weight.data = convatten_new.conv.weight.data.clone()

    out1 = convatten_new(ex)
    out2 = convatten_old(ex)
    out3 = convatten_claude(ex)
    out4 = mha(ex)

    print("new vs old:              ", torch.allclose(out1, out2))
    print("new vs claude:           ", torch.allclose(out1, out3))
    print("old vs claude:           ", torch.allclose(out2, out3))
    print()
    print("new vs mha:              ", torch.allclose(out1, out4))
    print("old vs mha:              ", torch.allclose(out2, out4))
    print("claude vs mha:           ", torch.allclose(out3, out4))

    print()
    print("Max diff (new vs old):   ", torch.max(torch.abs(out1 - out2)).item())
    print("Max diff (new vs claude):", torch.max(torch.abs(out1 - out3)).item())
    print("Max diff (old vs claude):", torch.max(torch.abs(out2 - out3)).item())

new vs old:               False
new vs claude:            True
old vs claude:            False

new vs mha:               True
old vs mha:               False
claude vs mha:            True

Max diff (new vs old):    1359523.5
Max diff (new vs claude): 0.0
Max diff (old vs claude): 1359523.5


# 2. Random and Spatial Sampling ConvNN Attention

In [10]:

"""Random Sampling ConvNN Attention Implementation"""
from torch import device


class MultiHeadConvNNAttention_Random(nn.Module):
    def __init__(self, 
                 d_hidden, 
                 num_heads,
                 attention_dropout, 
                 K, 
                 num_samples, 
                 convolution_type='depthwise',
                 seq_length=197):
        super(MultiHeadConvNNAttention_Random, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"
        
        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.num_samples = num_samples
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads

        if convolution_type == 'standard': 
            self.conv = nn.Conv1d(
                in_channels=self.in_channels,
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                bias=False
            )
        elif convolution_type == 'depthwise':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels, 
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                groups=self.in_channels, 
                bias=False
            )
        elif convolution_type == 'depthwise-separable':
            self.conv = nn.Sequential(
                # Depthwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.in_channels,
                    kernel_size=self.K,
                    stride=self.K,
                    padding=0,
                    groups=self.in_channels,
                    bias=False
                ), 
                # Pointwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0, 
                    bias=False
                )
            )
        self.conv.weight.data.fill_(1.0)
        
    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size() 
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def _prime_N(self, v, qk, K, rand_idx):
        # v: (B*num_heads, d_k, seq_length), qk: (B*num_heads, num_samples, seq_length)
        b, c, t = v.shape 
        topk_value, topk_indices = torch.topk(qk, k=K-1, dim=2, largest=True)

        # Map sample indices back to original matrix positions 
        mapped_tensor = rand_idx[topk_indices]
        token_indices = torch.arange(t, device=v.device).view(1, t, 1).expand(b, t, 1)
        final_indices = torch.cat([token_indices, mapped_tensor], dim=-1)
        topk_indices_exp = final_indices.unsqueeze(1).expand(b, c, t, K)

        # Expand topk values to match the shape of indices       
        topk_values_exp = topk_value.unsqueeze(1).expand(b, c, t, K-1)
        ones = torch.ones((b, c, t, 1), device=v.device)
        topk_values_exp = torch.cat((ones, topk_values_exp), dim=-1)

        # softmax
        topk_values_exp = torch.softmax(topk_values_exp, dim=-1)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime
    
    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        print("q|k|v shape: ", q.shape)

        # Random Sampling
        rand_idx = torch.randperm(x.shape[1], device=x.device)[:self.num_samples]
        k_sample = k[:, :, rand_idx, :]
        attn_matrix = torch.matmul(q, k_sample.transpose(-2, -1)) / np.sqrt(self.d_k) # (B, NH, SL, num_samples)

        print("attn scores shape: ", attn_matrix.shape)
        range_idx = torch.arange(len(rand_idx), device=q.device)
        print("range idx shape: ", range_idx.shape)
        print("rand idx shape: ", rand_idx.shape)
        attn_matrix[:, :, rand_idx, range_idx] = float('-inf')

        # Merge B and num_heads into dim for prime & conv
        ## (B, NH, SL, DK) → (B*NH, DK, SL) for v and (B, NH, num_samples, SL) → (B*NH, num_samples, SL)
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.num_samples)

        print("v_merged shape: ", v_merged.shape)
        print("am_merged shape: ", am_merged.shape)

        # Prime and Convolution 
        prime = self._prime_N(v_merged, am_merged, self.K, rand_idx)
        out = self.conv(prime) # (B*num_heads, d_k, seq_length
        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        output = self.W_o(self.combine_heads(out)) # (B, SL, d_hidden)
        return output

        
        


ex = torch.randn(3, 197, 768)
convnn_rand = MultiHeadConvNNAttention_Random(768, 3, 0.0, 9, 50, 'depthwise', 197)
out_rand = convnn_rand(ex)
print("Output shape (Random Sampling): ", out_rand.shape)

q|k|v shape:  torch.Size([3, 3, 197, 256])
attn scores shape:  torch.Size([3, 3, 197, 50])
range idx shape:  torch.Size([50])
rand idx shape:  torch.Size([50])
v_merged shape:  torch.Size([9, 256, 197])
am_merged shape:  torch.Size([9, 197, 50])
Output shape (Random Sampling):  torch.Size([3, 197, 768])


In [11]:
"""Spatial Sampling ConvNN Attention Implementation"""
class MultiHeadConvNNAttention_Spatial(nn.Module):
    def __init__(self, 
                 d_hidden, 
                 num_heads,
                 attention_dropout, 
                 K, 
                 num_samples, 
                 sample_padding,
                 convolution_type='depthwise',
                 seq_length=197):
        super(MultiHeadConvNNAttention_Spatial, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"
        
        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.num_samples = num_samples
        self.sample_padding = sample_padding
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads

        if convolution_type == 'standard': 
            self.conv = nn.Conv1d(
                in_channels=self.in_channels,
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                bias=False
            )
        elif convolution_type == 'depthwise':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels, 
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                groups=self.in_channels, 
                bias=False
            )
        elif convolution_type == 'depthwise-separable':
            self.conv = nn.Sequential(
                # Depthwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.in_channels,
                    kernel_size=self.K,
                    stride=self.K,
                    padding=0,
                    groups=self.in_channels,
                    bias=False
                ), 
                # Pointwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0, 
                    bias=False
                )
            )
        self.conv.weight.data.fill_(1.0)
        
    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size() 
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    
    def _prime_N(self, v, qk, K, rand_idx):
        # v: (B*num_heads, d_k, seq_length), qk: (B*num_heads, num_samples, seq_length)
        b, c, t = v.shape 
        topk_value, topk_indices = torch.topk(qk, k=K-1, dim=2, largest=True)

        # Map sample indices back to original matrix positions 
        mapped_tensor = rand_idx[topk_indices]
        token_indices = torch.arange(t, device=v.device).view(1, t, 1).expand(b, t, 1)
        final_indices = torch.cat([token_indices, mapped_tensor], dim=-1)
        topk_indices_exp = final_indices.unsqueeze(1).expand(b, c, t, K)

        # Expand topk values to match the shape of indices       
        topk_values_exp = topk_value.unsqueeze(1).expand(b, c, t, K-1)
        ones = torch.ones((b, c, t, 1), device=v.device)
        topk_values_exp = torch.cat((ones, topk_values_exp), dim=-1)

        # softmax
        topk_values_exp = torch.softmax(topk_values_exp, dim=-1)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime
    
    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        print("q|k|v shape: ", q.shape)

        # Random Sampling
        spat_idx = torch.linspace(0 + self.sample_padding, x.shape[1] - self.sample_padding - 1, self.num_samples, device=x.device).long()
        print(spat_idx)
        k_sample = k[:, :, spat_idx, :]
        attn_matrix = torch.matmul(q, k_sample.transpose(-2, -1)) / np.sqrt(self.d_k) # (B, NH, SL, num_samples)

        print("attn scores shape: ", attn_matrix.shape)
        range_idx = torch.arange(len(spat_idx), device=q.device)
        print("range idx shape: ", range_idx.shape)
        print("spat idx shape: ", spat_idx.shape)
        attn_matrix[:, :, spat_idx, range_idx] = float('-inf')

        # Merge B and num_heads into dim for prime & conv
        ## (B, NH, SL, DK) → (B*NH, DK, SL) for v and (B, NH, num_samples, SL) → (B*NH, num_samples, SL)
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.num_samples)

        print("v_merged shape: ", v_merged.shape)
        print("am_merged shape: ", am_merged.shape)

        # Prime and Convolution 
        prime = self._prime_N(v_merged, am_merged, self.K, spat_idx)
        out = self.conv(prime) # (B*num_heads, d_k, seq_length
        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out)
        output = self.W_o(self.combine_heads(out)) # (B, SL, d_hidden)
        return output

        
        


ex = torch.randn(3, 197, 768)
convnn_spat = MultiHeadConvNNAttention_Spatial(768, 3, 0.0, 9, 25, 1, "depthwise", 197)
out_spat = convnn_spat(ex)
print("Output shape (Spatial Sampling): ", out_spat.shape)

q|k|v shape:  torch.Size([3, 3, 197, 256])
tensor([  1,   9,  17,  25,  33,  41,  49,  57,  65,  73,  81,  89,  98, 106,
        114, 122, 130, 138, 146, 154, 162, 170, 178, 186, 195])
attn scores shape:  torch.Size([3, 3, 197, 25])
range idx shape:  torch.Size([25])
spat idx shape:  torch.Size([25])
v_merged shape:  torch.Size([9, 256, 197])
am_merged shape:  torch.Size([9, 197, 25])
Output shape (Spatial Sampling):  torch.Size([3, 197, 768])
